# 04. C++ 공유 라이브러리를 Python에서 사용하기 — 60분

목표: C++에서 `.so`를 만들고 Python의 ctypes로 배열·길이·결과 포인터를 전달합니다.
선행: 3교시의 CMake·참조·포인터, Python의 함수·리스트. 추가 Python 패키지는 필요하지 않습니다.

이 노트북은 **Python 커널** 에서 실제 C++ 도구를 호출합니다. GUI 없이 로컬·원격 모두 진행합니다.
코드는 아래 지정한 `exercise/` 파일을 JupyterLab 편집기로 수정합니다. 파일 저장 후 빌드 셀을 실행하세요.
노트북은 파일을 덮어쓰지 않습니다. 미완성 TODO의 검증 실패는 정상이며, 정답 확인 시점은 강사가 현장 상황에 맞춰 조절합니다.
정답은 별도 경로로 빌드하여 학습자 파일을 보존합니다. 빌드·결과는 호스트의 Data에 남습니다.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 이 노트북만으로 시작하기

열 파일은 `/workspace/03.C++/04_shared_library_python.ipynb`입니다. [03_memory_stl_cmake.ipynb의 「주소·참조·포인터와 수명」·「CMake」](03_memory_stl_cmake.ipynb)에 앞선 설명이 있으며 필요한 내용은 다음과 같습니다. `&value`는 주소, `*pointer`는 주소가 가리키는 값, `const double*`는 읽기 전용 배열 접근입니다. 주소만으로 배열 길이를 알 수 없으므로 길이도 전달합니다. CMake는 configure로 빌드 규칙을 만들고 build로 컴파일·링크하며 test로 결과를 검사합니다.

이전 센서 통계 `.a`나 Python 보고서는 필요하지 않습니다. 첫 환경 셀부터 실행하고 Jupyter 파일 편집기에서 `/workspace/03.C++/04_shared_library_python/exercise/sensor.cpp`를 수정·저장하세요. `print(read_text())`는 보기만 합니다. 현재 build 셀이 `/workspace/Data/build/03.C++/04_shared_library_python/exercise/libsensor.so`를 만들고 제공 wrapper가 새 Python 프로세스에서 읽습니다. 결과 JSON은 `/workspace/Data/outputs/03.C++/sensor_mean.json`입니다.
<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

Python의 편리한 실험 환경에서 C++ 함수를 재사용하려면 언어 사이에 값의 타입·길이·메모리 소유권을 약속해야 합니다. 이번 실습은 빠르다는 주장보다 서로 다른 언어가 같은 입력을 정확히 전달하고 오류를 구분하는 경계를 만드는 데 목적이 있습니다.

### 전체 과정에서 내 코드가 하는 일

```text
C++ 함수
  ↓
C ABI의 명시적 인수·상태 코드
  ↓
.so 빌드
  ↓
ctypes 타입 설정
  ↓
Python 호출·결과 비교
```

C++ 평균 계산과 오류 처리를 구현하고 Python 쪽 배열·길이·출력 공간을 맞춥니다. argtypes/restype는 주석이 아니라 실제 호출 계약입니다. 소스 변경 후 새 빌드와 새 프로세스 검증을 연결합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** C++가 실패 상태 코드를 돌려줬는데 평균 출력값을 사용해도 되나요?

**설명:** 안 됩니다. 상태 코드를 먼저 검사하고 성공했을 때만 출력 공간을 읽습니다. 포인터는 배열 길이나 유효 수명을 자동으로 알려주지 않습니다. Python과 C++가 같은 정상값·경계값에서 일치하는지도 확인합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–6절의 수업 핵심 경로** → [1일차 슬라이드 89쪽](https://docs.google.com/presentation/d/1pQvfu3XzIfFL9FFvJkP8a6iNKJMjQFGOQIboEF6DRjg/edit#slide=id.d1_s084) · [배포 PDF 89쪽](../슬라이드/1일차.pdf#page=89)에서 98쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 흐름 그림

![Python이 만든 입력·출력 버퍼를 C++이 호출 중 빌려 씁니다. 주소·길이·상태 코드·계산 결과를 각각 찾으세요.](../교재그림/cpp-buffer.svg)

Python이 만든 입력·출력 버퍼를 C++이 호출 중 빌려 씁니다. 주소·길이·상태 코드·계산 결과를 각각 찾으세요.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 제공된 빌드 틀에서 지정한 TODO를 수정하고 검증합니다.
- **단원 마지막 연결 질문:** Python 결과와 같음을 먼저 확인합니다. 기존 C++ 부품 재사용, 호출 비용, 수정·배포 부담 중 이 기능을 .so로 분리할 이유를 하나 선택합니다. C++이라는 이유만으로 빨라진다고 결론내리지 않습니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


## 처음 학습하는 학생을 위한 노트북 사용법

이 노트북의 설명은 위에서 아래로 읽되, 모든 문법을 외운 뒤 코드를 실행하려고 하지 않아도 됩니다. **코드 읽기 → 결과 예상 → 셀 실행 → 실제 결과 비교 → TODO 수정 → 검증** 순서로 반복하면 됩니다.

| 셀 또는 파일 | 하는 일 | 학생이 할 일 |
|---|---|---|
| Markdown 설명 | 새 용어와 코드의 목적 설명 | 낯선 용어를 표시하고 실행 결과를 먼저 예상 |
| 환경 Python 셀 | `g++`·CMake를 호출할 도우미와 경로 준비 | 내용을 바꾸지 말고 먼저 실행 |
| `examples/` | 완성된 작은 C++ 예제 | 출력 전에 변수 변화를 손으로 추적 |
| `exercise/` | 학생이 완성할 TODO | 지정된 부분만 수정하고 파일 저장 |
| build 셀 | 소스를 컴파일하고 링크 | 첫 오류부터 읽고 수정 후 다시 실행 |
| learner verification | 자신의 코드가 계약을 만족하는지 검사 | 정답 확인 시점은 강사가 조절 |
| 접힌 solution | 한 가지 참고 구현 | 통과 후 자신의 코드와 다른 이유를 비교 |

이 노트북의 커널은 Python입니다. Python 셀이 C++ 파일을 `g++` 또는 CMake에 전달하고 결과를 화면에 보여 줍니다. 마지막의 `CalledProcessError`만 읽지 말고, 그 위에 나온 컴파일러 또는 CTest의 **첫 번째 오류** 를 먼저 찾으세요. 공통 용어와 문제 해결 절차는 `03.C++/BEGINNER_GUIDE.md`에 더 자세히 정리되어 있습니다.

## 수업 필수와 복습·강사 시연 구분

| 구분 | 이 노트북에서 해당하는 절·셀 |
|---|---|
| **수업 필수** | 전체 호출 경로, C ABI 함수 서명, C++ TODO, `.so` 빌드, ctypes 타입 지정, ABI learner verification, JSON 결과 저장 |
| **복습·강사 시연** | name mangling·`nm -D` 세부 해석, NaN/Inf/overflow의 상세 원리, 오류 유형별 진단, 자습 확장과 참고 정답 비교 |

60분 수업에서는 Python→ctypes→`.so`→C++→Python 흐름을 한 번 완주합니다. 상세 ABI·오류 설명은 강사가 대표 사례를 시연하고 이후 복습에 사용합니다.
입력·합계의 유한성 검사도 필수 함수 계약에 포함됩니다. 강사 안내에 따라 정답을 참고해 완성할 수 있습니다. 수업에서는 검사 순서와 실패 시 출력 보존을 확인하고, 부동소수점의 상세 원리는 복습합니다.


## 진행표

| 시간 | 활동 |
|---|---|
| 0–10분 | 실행 파일·정적/공유 라이브러리·C ABI |
| 10–20분 | C++ 평균 함수 TODO |
| 20–30분 | CMake로 .so 빌드·심볼 확인 |
| 30–45분 | Python ctypes 호출·결과 비교 |
| 45–55분 | 경계·잘못된 입력 검증·정답 비교 |
| 55–60분 | Data 저장·재실행 |


In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '03.C++' / 'README.md').is_file())
LESSON = ROOT / '03.C++' / '04_shared_library_python'
BUILD = ROOT / 'Data' / 'build' / '03.C++' / '04_shared_library_python'
def run(args):
    result = subprocess.run([str(a) for a in args], text=True, capture_output=True, timeout=120)
    print(result.stdout, end='')
    if result.stderr: print(result.stderr, end='')
    result.check_returncode()
    return result

def build(variant):
    assert variant in ('exercise', 'solution')
    directory = BUILD / variant
    run(['cmake', '-S', LESSON, '-B', directory, '-DVARIANT='+variant, '-DCMAKE_BUILD_TYPE=Debug'])
    run(['cmake', '--build', directory, '--parallel', '2'])
    return directory
print('소스:', LESSON, '빌드:', BUILD)
run(['g++', '--version'])

import sys
print('Python:', sys.executable)


## 전체 호출 경로를 먼저 보기

<!-- procedure:lesson-06-1b4e67:start -->
![C++ 함수를 Python에서 호출하기](../교재그림/절차/lesson-06.svg)

재빌드한 so는 새 프로세스에서 다시 로드해 확인합니다. 반환 상태와 계산 결과를 구분하세요.
<!-- procedure:lesson-06-1b4e67:end -->


이 교시에서는 하나의 평균 계산이 여러 층을 통과합니다. 각 층의 책임을 구분하면 낯선 코드가 훨씬 읽기 쉬워집니다.

```text
Python 사용자 코드
    SensorLibrary.mean([20.0, 30.0, 25.0])
        ↓ Python 값을 C 호환 메모리로 변환
ctypes wrapper: sensor_api.py
        ↓ 함수 이름·인자 타입·주소·길이 전달
공유 라이브러리: libsensor.so
        ↓ sensor_mean(...) 기계어 실행
C++ 구현: exercise/sensor.cpp
        ↓ 상태 코드와 out_mean 결과
ctypes wrapper가 float 또는 Python 예외로 변환
```

Python은 C++ 소스 파일을 직접 읽어 실행하지 않습니다. 이미 ARM64 기계어로 빌드된 `.so`를 현재 Python 프로세스의 메모리에 로드하고, 공개된 함수 주소를 찾아 호출합니다. 그래서 소스를 고친 뒤 `.so`를 다시 빌드해야 하며, 이미 로드한 라이브러리는 새 프로세스나 커널 재시작으로 다시 읽는 것이 안전합니다.

이 경계에서는 Python 예외, C++ 예외, `std::vector` 같은 언어 전용 객체를 그대로 주고받지 않습니다. 단순한 숫자·포인터·길이·상태 코드로 좁은 계약을 만들어 두 언어가 같은 메모리 모양을 해석하게 합니다.

## 1. `.so`와 C ABI

공유 라이브러리는 main이 있는 실행 파일이 아닙니다. Python 프로세스가 로드하고 함수를 호출합니다.
C++은 오버로딩 등을 위해 심볼 이름을 변형합니다. `extern "C"`는 여기서 사용할 C 형태의 함수 이름을 노출합니다.
C++ 클래스·vector를 그대로 ctypes에 넘기지 않고 C 호환 타입의 포인터·길이·정수 상태를 사용합니다.

`int sensor_mean(const double* values, size_t n, double* out_mean)`에서:

- 입력 배열과 결과 메모리의 소유자는 Python입니다. C++은 포인터를 저장하거나 해제하지 않습니다.
- 반환값 0은 성공, 1은 null/빈 입력, 2는 비유한 입력·합계입니다.
- 실패 시 결과 메모리는 변경하지 않습니다.
- C++은 포인터만 보고 실제 배열 길이를 알 수 없습니다. Python wrapper가 만든 길이만 전달합니다. 실제보다 큰 길이를 실험하지 마세요.


In [ ]:
print((LESSON / 'include/sensor.h').read_text())


### 함수 선언을 한 조각씩 읽기

```cpp
int sensor_mean(const double* values, size_t n, double* out_mean);
```

| 조각 | 의미 | 이 함수의 계약 |
|---|---|---|
| 앞의 `int` | 반환 타입 | 계산값이 아니라 성공·실패 상태 코드 반환 |
| `sensor_mean` | 외부에 공개할 함수 이름 | ctypes가 같은 이름으로 찾음 |
| `const double* values` | 연속된 double 배열 첫 원소의 주소 | C++은 읽기만 하며 소유·해제하지 않음 |
| `size_t n` | 배열 원소 수 | 포인터 자체에는 길이 정보가 없어 별도 전달 |
| `double* out_mean` | 결과를 기록할 double 한 칸의 주소 | 성공한 경우에만 역참조해 결과 기록 |

포인터와 길이는 하나의 계약입니다. `values`가 null인데 먼저 `values[0]`을 읽거나, `n`이 실제 배열보다 크면 유효하지 않은 메모리에 접근할 수 있습니다. `out_mean`도 null일 수 있으므로 쓰기 전에 검사해야 합니다.

헤더의 `#ifdef __cplusplus`는 같은 헤더를 C++ 컴파일러가 읽을 때만 `extern "C"` 블록을 적용합니다. `extern "C"`는 함수 몸체를 C 언어로 바꾸지 않고, 외부 심볼 이름을 C 호환 방식으로 공개하라는 뜻입니다.

`size_t`는 메모리 크기와 개수를 표현하는 부호 없는 정수 타입입니다. Jetson의 64비트 프로세스에서는 일반적으로 64비트이므로 ctypes에서도 `c_size_t`로 맞춰야 합니다.

## 2. TODO — 평균 함수 구현

편집할 파일: **[03.C++/04_shared_library_python/exercise/sensor.cpp](04_shared_library_python/exercise/sensor.cpp)** .
1. null 포인터 또는 n==0이면 1을 반환합니다.
2. 값을 누적할 때 각 값과 합계가 `std::isfinite`인지 확인합니다. 아니면 2입니다.
3. 모든 검사가 성공한 경우에만 평균을 `*out_mean`에 쓰고 0을 반환합니다.

힌트: 배열 접근 전 인자를 검사합니다. 빈 입력에서 0으로 나누지 않도록 합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `sensor_mean(const double* values, size_t n, double* out_mean)`의 입력은 배열 주소·개수·결과를 쓸 주소입니다. 함수 반환값은 평균이 아니라 상태 코드 `0/1/2`이며 평균은 `*out_mean`에 기록합니다. `{20,30,25}`, n=3이면 상태 `0`, 평균 `25.0`입니다. 정확한 선언은 함께 제공된 헤더를 유지하세요.
**경계:** `values`와 `out_mean` 어느 쪽이든 null이거나 n=0이면 `1`, 입력 원소 또는 누적 합이 유한하지 않으면 `2`입니다. 실패할 때 기존 출력값을 바꾸지 않습니다.
**작성 흐름 힌트:** 포인터·길이 검사 → 지역 변수 sum에 누적하면서 유한성 검사 → 마지막에 한 번만 출력 주소에 평균 쓰기 → 성공 코드 반환. 배열을 할당·해제하는 책임은 호출자에게 있습니다.
**설명할 지점:** “return 0은 평균 0이 아니라 계산 성공입니다. 결과를 마지막에 쓰면 실패한 계산으로 기존 출력이 오염되지 않습니다.”
<!-- todo-review:requirements:end -->

**구현 범위와 성공 검사:** 호출자가 `n`개의 읽을 수 있는 double과 쓸 수 있는 double 한 칸을 준비했다고 가정하고, 함수 안에서 새 메모리를 할당·해제하지 않습니다. 실패 경로에서 `*out_mean=0` 같은 초기화도 하지 않습니다. `std::isfinite`는 NaN·양/음의 무한대에 false이며, 원소가 각각 유한해도 더한 합계가 범위를 넘으면 코드 2입니다. 빈 입력/두 null 경계/NaN/무한대/합계 overflow/실패 시 출력 보존을 아래 `tests/test_sensor.py`가 확인합니다. 이 검사와 마지막 Python 평균 비교·JSON 저장까지 통과하면 완료입니다.


In [ ]:
print((LESSON / 'exercise/sensor.cpp').read_text())


### 실패해도 출력값을 보존하는 이유

이 함수는 계산 결과와 오류를 서로 다른 통로로 전달합니다.

```text
반환값 int ──> 호출 성공 여부
out_mean 주소 ──> 성공했을 때의 실제 평균
```

함수 초반에 임시 합계와 입력 유효성을 확인한 뒤 모든 검사가 성공했을 때만 출력 메모리를 바꾸면, 호출자는 실패 상태에서 오래된 값이나 일부 계산값을 새 결과로 오해하지 않습니다. 테스트는 출력 변수를 미리 알려진 값으로 채우고 오류 호출 뒤 그대로 남는지 확인합니다.

`NaN`은 숫자 연산 결과가 유효한 실수가 아님을 나타내며, `Inf`는 표현 범위를 넘어선 무한값입니다. 개별 입력이 유한해도 매우 큰 값을 더한 합계가 무한값이 될 수 있으므로 입력과 누적 결과는 별도 경계입니다. 이 검사는 단순 평균 알고리즘보다 **언어 경계에서 잘못된 값을 명시적으로 보고하는 계약** 을 배우기 위한 것입니다.

## 3. 빌드와 심볼 확인

`add_library(sensor SHARED ...)`는 위치 독립 코드를 사용하여 `libsensor.so`를 생성합니다.
Debug 빌드를 사용합니다. Python wrapper는 절대 경로로 이 파일을 열어 다른 라이브러리를 잘못 읽지 않도록 합니다.


In [ ]:
print((LESSON / 'CMakeLists.txt').read_text())
exercise_build = build('exercise')
run(['nm', '-D', exercise_build / 'libsensor.so'])


### 빌드 로그와 `nm -D` 결과 읽기

CMake의 `add_library(sensor SHARED ...)`는 Linux 공유 라이브러리 `libsensor.so`를 만듭니다. 이름 앞의 `lib`와 확장자 `.so`는 플랫폼 관례이며, CMake target 이름은 `sensor`입니다.

`nm -D libsensor.so`는 동적 심볼 표를 보여 줍니다. 여러 줄 중 `sensor_mean`이라는 읽을 수 있는 이름을 찾으세요. 보인다면 Python 로더가 이름으로 함수를 찾을 수 있다는 뜻입니다. `extern "C"`가 빠지면 C++ 매개변수 정보를 담은 변형된 이름이 보일 수 있습니다. 심볼이 보인다고 함수의 계산이 올바르다는 뜻은 아니므로 이후 ABI 테스트가 필요합니다.

빌드 문제는 다음처럼 나눕니다.

| 증상 | 의미 | 확인할 항목 |
|---|---|---|
| `sensor.h: No such file` | 컴파일러가 헤더를 못 찾음 | `target_include_directories`, 파일 경로 |
| 컴파일 오류 | C++ 문법·타입 문제 | 첫 `error:`와 표시된 소스 줄 |
| `.so`가 없음 | build가 끝나지 않았거나 다른 폴더를 봄 | exercise build 경로와 이전 로그 |
| `sensor_mean` 심볼이 없음 | 외부 이름 또는 소스 연결 문제 | `extern "C"`, 올바른 variant, `nm -D` |
| `wrong ELF class` 등 | 다른 CPU/비트수용 바이너리 | Jetson ARM64 컨테이너 안에서 다시 빌드 |

## 4. ctypes에서 자료형 지정

C++ `double*`는 `POINTER(c_double)`, `size_t`는 `c_size_t`, 상태 `int`는 `c_int`로 지정합니다.
`argtypes`와 `restype`를 생략해 우연히 돌아가는 호출에 의존하지 않습니다.
`(c_double * n)(*values)`가 배열을 만들고 `byref(output)`이 결과 변수의 주소를 전달합니다.
배열·결과 변수는 함수가 반환할 때까지 Python 지역 변수로 살아 있습니다.


In [ ]:
print((LESSON / 'python/sensor_api.py').read_text())


### Python wrapper를 위에서 아래로 추적하기

`SensorLibrary`는 위험할 수 있는 저수준 호출을 한곳에 감싸 사용자가 평범한 Python 함수처럼 쓰게 합니다.

1. `Path(...).resolve(strict=True)`는 실제 존재하는 `.so`의 절대 경로를 얻습니다. 파일이 없으면 여기서 명확히 실패합니다.
2. `ct.CDLL(...)`은 공유 라이브러리를 현재 Python 프로세스에 로드합니다.
3. `argtypes`는 세 인자를 `double*`, `size_t`, `double*`에 대응시킵니다.
4. `restype`는 반환 상태가 C의 `int`임을 지정합니다.
5. `values = list(values)`는 generator 등 반복 가능한 입력을 한 번 고정하고 길이와 내용이 달라지지 않게 합니다.
6. `(ct.c_double * len(values))(*values)`는 Python 숫자를 연속된 C double 배열로 복사합니다.
7. `ct.c_double()`은 C++이 결과를 쓸 한 칸을 만들고 0.0으로 초기화합니다.
8. `ct.byref(output)`은 그 한 칸의 주소를 전달합니다.
9. 상태 코드를 검사해 알려진 실패는 `ValueError`, 알 수 없는 상태는 `RuntimeError`로 바꿉니다.
10. 성공했을 때만 `output.value`를 Python `float`로 돌려줍니다.

호출 중에는 `array`와 `output` 지역 변수가 살아 있으므로 C++ 포인터가 유효합니다. C++이 이 주소를 전역 변수에 저장해 함수 반환 뒤 다시 사용하면 Python 객체 수명을 보장할 수 없어 위험합니다. 그래서 헤더 계약에 '포인터를 저장하지 않는다'고 명시합니다.

`argtypes`를 잘못 쓰면 Python이 친절한 타입 오류를 낼 수도 있지만, 메모리 주소를 잘못 해석해 Python 프로세스 자체가 종료될 수도 있습니다. ABI 경계에서는 양쪽 선언을 눈으로 대조하고 자동 테스트를 실행해야 합니다.

### 학습자 코드 실행·검증

아래 검사는 **exercise의 libsensor.so** 를 새 Python 프로세스에서 로드합니다.
대표값·음수·빈 입력·NaN·Inf·합계 overflow·null 포인터·실패 시 결과 보존을 확인합니다.


In [ ]:
run([sys.executable, LESSON / 'tests/test_sensor.py', exercise_build / 'libsensor.so'])
print('learner_verification=PASS')


### ABI 테스트가 확인하는 경계

검증은 '평균이 한 번 맞다'보다 넓은 계약을 확인합니다.

| 범주 | 대표 검사 | 찾으려는 문제 |
|---|---|---|
| 정상 입력 | 여러 값, 한 값, 음수, 임의 값 97개 | 합계·나눗셈·길이 처리 |
| 입력 보존 | 호출 전후 Python list와 C 배열 비교 | `const` 입력을 실수로 변경했는지 |
| 빈/비유한 입력 | 빈 배열, NaN, Inf, overflow | 상태 코드와 Python 예외 변환 |
| null 포인터 | 입력 또는 출력 주소가 없음 | 역참조 전 인자 검사 |
| 실패 원자성 | 출력에 123.0을 넣고 실패 호출 | 실패할 때 출력 메모리를 바꾸지 않는지 |
| ABI 모양 | ctypes로 포인터·길이 직접 호출 | wrapper 밖에서도 C 함수 계약이 맞는지 |

검증 실패를 읽을 때 Python traceback의 마지막 `AssertionError`만 보지 말고 바로 위의 호출과 테스트 줄을 확인하세요. 프로세스가 `Segmentation fault`로 끝나면 보통 포인터 검사 순서, 전달 길이, `argtypes`를 우선 살펴봅니다. 예외 메시지가 예상과 다르면 C++ 상태 코드와 wrapper의 분기 대응을 함께 확인합니다.

강사가 안내하는 시점에 다음 참고 정답 셀을 펼칩니다. 정답의 문장 모양보다 검사 순서, 임시값 사용, 출력 기록 시점이 계약을 어떻게 지키는지 비교하세요.

### 참고 정답과 독립 검증

강사가 안내하는 시점에 확인하고 자신의 코드로 재검증합니다. 정답은 별도 디렉터리에서 빌드합니다.


In [ ]:
print((LESSON / 'solution/sensor.cpp').read_text())
solution_build = build('solution')


### 정답 해설 — 입력부터 결과까지

**한 번의 호출을 따라가기:** Python `[20.0,30.0,25.0]`은 그대로 C++ 배열이 아닙니다. wrapper가 연속된 `c_double` 배열을 만들고, 첫 주소·원소 수 3·결과 한 칸의 주소를 전달합니다. C++ 함수와 Python은 같은 프로세스 안에서 이 버퍼를 사용합니다.

```text
Python이 소유하는 한 프로세스의 버퍼
입력: [20.0][30.0][25.0]       출력: [아직 결과 아님]
         │ 첫 주소, n=3                    ▲ out_mean 주소
         └──────── sensor_mean ────────────┘
                      │
                int 상태 0/1/2 → wrapper가 먼저 확인
```

1. `values == nullptr`, `out_mean == nullptr`, `n == 0`을 먼저 검사합니다. 유효하지 않은 주소를 읽기 전에 상태 1로 끝냅니다.
2. 지역 합계에 입력을 더합니다. 각 값과 누적 합계가 유한한지 `std::isfinite`로 확인하고 실패하면 상태 2입니다. 유한한 큰 값끼리 더해도 overflow가 생길 수 있어 두 검사가 필요합니다.
3. 모든 검사를 지난 뒤에만 `*out_mean = sum / n`으로 결과를 씁니다. `*`는 그 주소의 한 칸에 쓰는 역참조입니다. 합계 75, 개수 3이므로 평균 25.0입니다.
4. `return 0`은 평균 0이라는 뜻이 아니라 성공 상태입니다. wrapper는 성공일 때 `output.value`를 읽고 오류 상태는 예외로 바꿉니다.

**경계 추적:** 출력에 123.0이 있던 호출에 빈 입력이나 NaN을 전달하면 실패 상태만 반환하고 출력은 123.0 그대로여야 합니다. 그래서 함수 초반에 출력부터 0으로 지우지 않습니다. 실패한 출력은 새 평균으로 사용하지 않습니다.

`argtypes`의 `POINTER(c_double), c_size_t, POINTER(c_double)`과 `restype=c_int`는 실제 함수 서명과 맞춰야 합니다. 입력·출력 버퍼는 Python이 소유하고 C++은 호출 동안만 빌리므로 주소를 저장하거나 해제하지 않습니다.

**확인 질문:** 상태 코드와 출력 포인터를 하나로 혼동하면 어떤 값이 평균처럼 보이나요? 코드를 외우기보다 검사 순서·소유권·실패 시 보존을 설명하세요.

**적용·재검증:** 자신의 답안을 저장한 뒤 지정된 exercise 파일/작성 셀에 반영하고 저장→exercise 빌드→학습자 검증 순서로 실행합니다. solution 빌드는 별도 참고 결과입니다. 오류는 첫 compiler/CTest 메시지부터 읽습니다. `.so`를 이미 로드했다면 재빌드 후 새 Python 프로세스 또는 커널 재시작과 환경 셀 재실행으로 다시 로드합니다. 결과는 기존 Data 경로에 저장합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](04_shared_library_python/solution/sensor.cpp)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```cpp
#include "sensor.h"
#include <cmath>
extern "C" int sensor_mean(const double* values, size_t n, double* out_mean) {
    if (values == nullptr || out_mean == nullptr || n == 0) return 1;
    double sum{0.0};
    for (size_t i=0; i<n; ++i) {
        if (!std::isfinite(values[i])) return 2;
        sum += values[i];
        if (!std::isfinite(sum)) return 2;
    }
    *out_mean = sum / static_cast<double>(n);
    return 0;
}
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
run(['ctest', '--test-dir', solution_build, '--output-on-failure'])
print('solution_verification=PASS')


## 5. 자신의 결과를 Data에 저장

Python의 `sum/len`과 자신의 C++ 평균을 비교합니다. 아래 프로그램은 solution이 아닌 exercise 라이브러리를 사용합니다.


In [ ]:
run([sys.executable, LESSON / 'python/demo.py', exercise_build / 'libsensor.so', '--output', ROOT / 'Data/outputs/03.C++/sensor_mean.json'])


## 6. 재빌드·재생성과 흔한 문제

- 파일을 저장하지 않았다면 빌드는 이전 소스를 사용합니다.
- `.so`를 Python 커널에서 한 번 로드한 뒤 같은 경로로 재빌드해도 새 코드로 바뀐다고 보장할 수 없습니다. **새 Python 프로세스 또는 커널 재시작** 후 로드합니다. 이 노트북은 매 호출을 새 프로세스에서 수행합니다.
- `OSError`: 경로·빌드 성공·ARM64 여부를 확인합니다. `undefined symbol`: 헤더·extern C·링크를 확인합니다.
- 컨테이너 삭제 후 재생성해도 Data의 `.so`와 결과는 남습니다. 이미지/컴파일 환경이 바뀌면 다시 빌드하세요.

## 자습 확장 (20–30분)

`python/demo.py`의 values를 음수·한 개 값으로 바꾸어 실행하세요. 빈 배열에서는 ValueError가 나오는지 확인합니다.
정답 기대값은 `sum(values)/len(values)`이고 빈 배열은 평균을 정의하지 않습니다.
결과 JSON 경로를 별도 이름으로 지정하여 원래 결과를 보존하세요.

## 완료 조건

자신이 구현한 `.so`가 ABI 검증을 통과하고 `Data/outputs/03.C++/sensor_mean.json`에 Python과 일치하는 결과가 저장됩니다.
포인터를 누가 소유하는지, 왜 argtypes/restype가 필요한지, 왜 재빌드 후 새 프로세스를 사용하는지 설명하세요.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 자기 .so 빌드·ctypes 검증·결과 JSON 저장

[1일차 슬라이드 89쪽](https://docs.google.com/presentation/d/1pQvfu3XzIfFL9FFvJkP8a6iNKJMjQFGOQIboEF6DRjg/edit#slide=id.d1_s084) · [배포 PDF 89쪽](../슬라이드/1일차.pdf#page=89)로 돌아가 **89–98쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

